# Ollama - Hands-On Tour

A single notebook walking through `../docs/`: health check &rarr; native
API calls &rarr; OpenAI-compatible comparison &rarr; a custom model built
via the API &rarr; model specs &rarr; GPU placement &rarr; context window
memory cost &rarr; streaming timing. Every cell was run for real while
building this module.

## Setup

```bash
ollama pull llama3.1:8b
ollama pull nomic-embed-text
pip install requests openai
```

In [ ]:
import requests

BASE_URL = "http://localhost:11434"
MODEL = "llama3.1:8b"

response = requests.get(f"{BASE_URL}/api/tags", timeout=5)
print("Ollama reachable:", response.status_code == 200)
for m in response.json()["models"]:
    print(f"  {m['name']:<28} {m['size']/1_000_000_000:.2f} GB")

## 1. Native API calls

See `docs/04-The-Native-REST-API.md`. `/api/generate`, `/api/chat`,
`/api/embeddings` - the raw calls every client library wraps.

In [ ]:
result = requests.post(f"{BASE_URL}/api/chat", json={
    "model": MODEL, "messages": [{"role": "user", "content": "Say OK"}], "stream": False,
}).json()

print("message:", result["message"])
print(f"load: {result['load_duration']/1e9:.3f}s, generation: {result['eval_duration']/1e9:.3f}s")

## 2. OpenAI-compatible vs. native - same server

See `docs/05-The-OpenAI-Compatible-Endpoint.md`.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="ollama")

prompt = "What is a Kubernetes readiness probe? One sentence."

via_openai = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0,
).choices[0].message.content

via_native = requests.post(f"{BASE_URL}/api/chat", json={
    "model": MODEL, "messages": [{"role": "user", "content": prompt}],
    "options": {"temperature": 0}, "stream": False,
}).json()["message"]["content"]

print("openai-compatible:", via_openai)
print("native:           ", via_native)

## 3. Build a custom model via the API

See `docs/06-Modelfiles-Customizing-a-Model.md`. No Modelfile text
file needed - `/api/create` accepts the same configuration directly.

In [ ]:
CUSTOM_MODEL = "terse-devops-notebook"

response = requests.post(f"{BASE_URL}/api/create", json={
    "model": CUSTOM_MODEL,
    "from": MODEL,
    "system": "You are a terse DevOps assistant. Answer in one sentence, no exceptions.",
    "parameters": {"temperature": 0},
}, stream=True)
for _ in response.iter_lines():
    pass  # drain build status messages

question = "What is a Kubernetes readiness probe?"

base_answer = requests.post(f"{BASE_URL}/api/chat", json={
    "model": MODEL, "messages": [{"role": "user", "content": question}], "stream": False,
}).json()["message"]["content"]

custom_answer = requests.post(f"{BASE_URL}/api/chat", json={
    "model": CUSTOM_MODEL, "messages": [{"role": "user", "content": question}], "stream": False,
}).json()["message"]["content"]

print("base model answer length:  ", len(base_answer), "chars")
print("custom model answer length:", len(custom_answer), "chars")
print("\ncustom model's actual answer:", custom_answer)

requests.delete(f"{BASE_URL}/api/delete", json={"model": CUSTOM_MODEL})
print("\ncleaned up.")

## 4. Real model specs

See `docs/07-Model-Formats-and-Quantization.md`. `nomic-embed-text`
should report only `embedding` capability - no `completion`.

In [ ]:
for name in [m["name"] for m in requests.get(f"{BASE_URL}/api/tags").json()["models"]]:
    info = requests.post(f"{BASE_URL}/api/show", json={"model": name}).json()
    details = info.get("details", {})
    print(f"{name:<28} params={details.get('parameter_size','?'):<8} quant={details.get('quantization_level','?'):<8} capabilities={info.get('capabilities')}")

## 5. GPU vs. CPU placement

See `docs/09-GPU-vs-CPU-How-Ollama-Chooses.md`.

In [ ]:
requests.post(f"{BASE_URL}/api/generate", json={"model": MODEL, "prompt": "hi", "stream": False})
ps = requests.get(f"{BASE_URL}/api/ps").json()["models"]
for m in ps:
    gpu_pct = round((m["size_vram"] / m["size"]) * 100) if m["size"] else 0
    print(f"{m['name']}: {gpu_pct}% GPU")

## 6. Context window memory cost

See `docs/10-Context-Length-and-Memory.md`. Verified: a larger
`num_ctx` measurably increases reported memory for the SAME short
prompt.

In [ ]:
for num_ctx in [4096, 32768]:
    requests.post(f"{BASE_URL}/api/generate", json={"model": MODEL, "keep_alive": 0})
    requests.post(f"{BASE_URL}/api/generate", json={
        "model": MODEL, "prompt": "hi", "options": {"num_ctx": num_ctx}, "stream": False,
    })
    info = next(m for m in requests.get(f"{BASE_URL}/api/ps").json()["models"] if m["name"] == MODEL)
    print(f"num_ctx={num_ctx:>6}: {info['size']/1_000_000_000:.2f} GB")

## 7. Streaming: time-to-first-token vs. total time

See `docs/13-Streaming-Responses.md`.

In [ ]:
import json as json_module
import time

start = time.time()
first_token_time = None

response = requests.post(f"{BASE_URL}/api/generate", json={
    "model": MODEL, "prompt": "List 5 benefits of Infrastructure as Code.", "stream": True,
}, stream=True)

for line in response.iter_lines():
    if not line:
        continue
    chunk = json_module.loads(line)
    if first_token_time is None and chunk.get("response"):
        first_token_time = time.time()
    if chunk.get("done"):
        break

print(f"Time to first token: {first_token_time - start:.2f}s")
print(f"Total time: {time.time() - start:.2f}s")

## Wrap-up

| Section | Concept | Docs chapter |
| --- | --- | --- |
| 1 | Native API | 04 |
| 2 | OpenAI-compatible endpoint | 05 |
| 3 | Modelfiles (via API) | 06 |
| 4 | Model specs and capabilities | 07 |
| 5 | GPU vs. CPU | 09 |
| 6 | Context window memory | 10 |
| 7 | Streaming and TTFT | 13 |

Not covered here but worth running from `../examples/`:
`09_keep_alive_tuning.py` (a real timing quirk found while building
this module - `keep_alive=0` isn't instant when combined with a real
prompt) and `10_performance_diagnostics.py` (the full "why is this
slow" checklist).

Next: `09-LangChain` - a framework layer on top of everything this
module just showed happening directly.